## 1. Setup

Import the project code and define the local paths you want to use.

# Conformal Risk Control Template

A clean notebook template for conformal risk control on top of a trained NanoDet detector.

Fill in the placeholders with your own prediction files, ground-truth files, and checkpoint paths.

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("<path/to/Modified-NanoDet-Plus>")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from tools.crc import ConformalDetectionCRC
from nanodet.conformal import load_pred_gt_data

print(f"Project root: {PROJECT_ROOT}")

## 2. Generate or Load JSON Inputs

The calibration and test prediction JSON files should be generated with `tools/get_full_scores.py`.
The ground-truth COCO JSON files are loaded alongside them for matching and evaluation.

In [ ]:
CAL_PRED_JSON = "<path/to/calibration_scores_generated_by_get_full_scores.json>"
CAL_GT_JSON = "<path/to/calibration/ground_truth.json>"
TEST_PRED_JSON = "<path/to/test_scores_generated_by_get_full_scores.json>"
TEST_GT_JSON = "<path/to/test/ground_truth.json>"

cal_dict = load_pred_gt_data(CAL_PRED_JSON, CAL_GT_JSON)
test_dict = load_pred_gt_data(TEST_PRED_JSON, TEST_GT_JSON)

calibration_data = list(cal_dict.values())
test_data = list(test_dict.values())

print(f"Calibration : {len(calibration_data):4d} images | "
      f"{sum(len(s['detections']) for s in calibration_data):6d} detections")
print(f"Test        : {len(test_data):4d} images | "
      f"{sum(len(s['detections']) for s in test_data):6d} detections")

## 3. Set Up the CRC Pipeline

Configure the three conformal stages and the risk targets here.

In [ ]:
detector = ConformalDetectionCRC(
    calibration_data=calibration_data,
    alpha_cnf=0.01,
    alpha_loc=0.10,
    alpha_cls=0.05,
    required_coverage=0.90,
    nonconformity_type='margin',
    mode='relative-miss-rate',
    miss_rate=0.10,
    lambda_mode='direct-inverted',
)

detector.run_threshold_stage(verbose=True)
detector.run_localization_stage(verbose=True)
detector.run_classification_stage(verbose=True)
detector.print_summary()

## 4. Apply and Validate

Apply the fitted pipeline to the test split and check that the empirical guarantees hold.

In [ ]:
conformal_predictions = detector.apply(test_data, verbose=True)
validation = detector.validate_results(test_data)

detector.print_test_summary(validation)

print("\nAll guarantees met:", "YES" if all([
    validation['threshold_guarantee'],
    validation['localization_guarantee'],
    validation['classification_guarantee'],
]) else "NO")

## Notes

Use the prediction JSON files produced by `tools/get_full_scores.py` together with the matching COCO ground-truth files.